# 5부. 딥러닝 모델링과 평가

장별 완성 예제. 학습용 가상 데이터이며 공식 시험 제출 형식이 아니다. examples 폴더에서 전체 실행한다.

In [ ]:
from pathlib import Path
import os
if not Path('lab.py').is_file() and Path('../lab.py').is_file():
    os.chdir('..')
assert Path('lab.py').is_file(), 'Run with the examples folder and data present'
get_ipython().run_line_magic('matplotlib', 'inline')


## 22장. TensorFlow·Keras 시작하기

입력 형태·Dense 계층·학습 설정을 확인한다.

In [ ]:
import numpy as np
from lab import dense_data, keras_model, fit_dense, dense_predict

X_train, X_val, X_test, y_train, y_val, y_test, prep = dense_data()
assert X_train.dtype == np.float32
model = keras_model(X_train.shape[1])
model.summary()
history = fit_dense(model,X_train,y_train,X_val,y_val,epochs=2)
probability = dense_predict(model,X_val)
assert probability.shape == (120,)
assert np.isfinite(probability).all()
assert "val_loss" in history.history
print("Input features:",X_train.shape[1])

## 23장. 딥러닝 분류 모델 만들기

출력층·라벨·손실의 조합을 맞춘다.

In [ ]:
from lab import dense_data, fit_dense, dense_predict
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score

X_train, X_val, X_test, y_train, y_val, y_test, prep = dense_data()
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32,activation="relu"),
    tf.keras.layers.Dense(16,activation="relu"),
    tf.keras.layers.Dense(1,activation="sigmoid"),
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
              loss="binary_crossentropy",metrics=["accuracy"])
history = fit_dense(model,X_train,y_train,X_val,y_val,epochs=12)
prob = dense_predict(model,X_val)
pred = (prob >= 0.5).astype(int)
assert model.output_shape[-1] == 1
print(accuracy_score(y_val,pred),f1_score(y_val,pred,zero_division=0))

## 24장. 딥러닝 회귀 모델 만들기

연속 타깃의 학습과 원래 단위의 평가를 수행한다.

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from lab import dense_data, keras_model, fit_dense, dense_predict

X_train, X_val, X_test, y_train, y_val, y_test, prep = dense_data("regression")
mean, std = y_train.mean(), y_train.std()
assert std > 0
train_scaled = (y_train-mean)/std
val_scaled = (y_val-mean)/std
model = keras_model(X_train.shape[1],task="regression")
history = fit_dense(model,X_train,train_scaled,X_val,val_scaled,epochs=12)
pred = dense_predict(model,X_val)*std+mean
assert pred.shape == y_val.shape
assert np.isfinite(pred).all()
print("MAE hours:",mean_absolute_error(y_val,pred))
print("RMSE hours:",root_mean_squared_error(y_val,pred))

## 25장. 학습 곡선과 성능 개선하기

검증 손실·EarlyStopping·Dropout을 활용한다.

In [ ]:
import matplotlib.pyplot as plt
from lab import dense_data, keras_model, fit_dense
import tensorflow as tf

X_train, X_val, X_test, y_train, y_val, y_test, prep = dense_data()
model = keras_model(X_train.shape[1],dropout=0.2)
early = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=2,
                                         restore_best_weights=True)
history = fit_dense(model,X_train,y_train,X_val,y_val,epochs=20,callbacks=[early])
assert len(history.history["loss"]) <= 20
assert len(history.history["loss"]) == len(history.history["val_loss"])
plt.plot(history.history["loss"],label="train")
plt.plot(history.history["val_loss"],label="validation")
plt.xlabel("Epoch index")
plt.ylabel("Loss")
plt.legend()
plt.show()
plt.close("all")
print("Completed epochs:",len(history.history["loss"]))